In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# --- Hyperparameters ---
PATCH_SIZE = 5
NUM_PATCHES = (75 // PATCH_SIZE) ** 2
EMBED_DIM = 64
NUM_BLOCKS = 2

import tensorflow_datasets as tfds

# --- Constants for new dataset ---
IMG_SIZE = 75
NUM_CLASSES = 8

# --- Load & preprocess Colorectal Histology ---
def preprocess(example):
    image = tf.image.resize(example['image'], [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    label = example['label']
    return image, label

ds = tfds.load("colorectal_histology", split="train", as_supervised=False)

# Shuffle, preprocess, and split manually
ds = ds.shuffle(5000, reshuffle_each_iteration=True).map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

# Split: 80% train, 10% val, 10% test
train_ds = ds.take(4000).batch(64).prefetch(tf.data.AUTOTUNE)
val_ds   = ds.skip(4000).take(500).batch(64).prefetch(tf.data.AUTOTUNE)
test_ds  = ds.skip(4500).take(500).batch(64).prefetch(tf.data.AUTOTUNE)

# --- Cheap Operations Dictionary ---
def cheap_ops_dict(embed_dim):
    return {
        "identity": layers.Lambda(lambda x: x),
        "softmax_gate": layers.Lambda(lambda x: x * tf.nn.softmax(x, axis=1)),
        "zeros": layers.Lambda(lambda x: tf.zeros_like(x)),
        "noise_injection": layers.Lambda(lambda x: x + tf.random.normal(tf.shape(x), stddev=0.01)),
        "spatial_dropout": layers.SpatialDropout1D(0.1),
        "scale_small": layers.Lambda(lambda x: 0.1 * x),
        "scale_large": layers.Lambda(lambda x: 10.0 * x),
        "negate": layers.Lambda(lambda x: -x),
        "abs": layers.Lambda(lambda x: tf.abs(x)),
        "square": layers.Lambda(lambda x: tf.square(x)),
        "sqrt": layers.Lambda(lambda x: tf.sqrt(tf.abs(x) + 1e-5)),
        "sign": layers.Lambda(lambda x: tf.sign(x)),
        "binarize": layers.Lambda(lambda x: tf.cast(x > 0.0, tf.float32)),
        "tanh_gate": layers.Lambda(lambda x: x * tf.tanh(x)),
        "sigmoid_gate": layers.Lambda(lambda x: x * tf.nn.sigmoid(x)),
        "mean_subtract": layers.Lambda(lambda x: x - tf.reduce_mean(x, axis=1, keepdims=True)),
        "std_norm": layers.Lambda(lambda x: (x - tf.reduce_mean(x, axis=1, keepdims=True)) /
                                               (tf.math.reduce_std(x, axis=1, keepdims=True) + 1e-5)),
        "global_context": layers.Lambda(lambda x: tf.repeat(tf.reduce_mean(x, axis=1, keepdims=True),
                                                            repeats=tf.shape(x)[1], axis=1)),
        "reverse": layers.Lambda(lambda x: tf.reverse(x, axis=[1])),
        "roll": layers.Lambda(lambda x: tf.roll(x, shift=1, axis=1)),
        #"channel_shuffle": layers.Lambda(lambda x: tf.random.shuffle(tf.transpose(x, perm=[0,2,1])) if tf.executing_eagerly() else x),
        "slice_first_half": layers.Lambda(lambda x: x[:, :, :x.shape[-1] // 2]),
        "slice_second_half": layers.Lambda(lambda x: x[:, :, x.shape[-1] // 2:]),
        "linear_skip": models.Sequential([
            layers.Dense(embed_dim),
            layers.Activation('linear')
        ]),
    }

# --- Helper: Auto-balance Head Dims ---
def auto_balance_ops(op_names, embed_dim):
    n = len(op_names)
    base = embed_dim // n
    rem = embed_dim % n
    return [(name, base + (1 if i < rem else 0)) for i, name in enumerate(op_names)]

# --- Patch Embedding ---
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size, embed_dim):
        super().__init__()
        self.projection = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size, padding='same')
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        return self.flatten(self.projection(x))

# --- Multi-Head CheapOp Block ---
class MultiHeadCheapOpBlock(layers.Layer):
    def __init__(self, embed_dim, op_specs, projection=True):
        super().__init__()
        self.embed_dim = embed_dim
        self.projection = projection
        self.pre_ln = layers.LayerNormalization(epsilon=1e-6)
        #self.conv = layers.Conv2D(embed_dim, kernel_size=5, padding='same', activation='relu')
        self.conv = layers.DepthwiseConv2D(kernel_size=5, padding='same', activation='gelu')
        self.cheap_heads = [cheap_ops_dict(head_dim)[name] for name, head_dim in op_specs]
        self.output_proj = layers.Dense(embed_dim) if projection else None
        self.residual_gate = self.add_weight(shape=(), initializer="ones", trainable=True, name="residual_gate")

    def call(self, x):
        B = tf.shape(x)[0]
        N = tf.shape(x)[1]  # Actual number of patches
        D = x.shape[-1]

        # Infer spatial dimensions dynamically (assume square)
        H = tf.cast(tf.sqrt(tf.cast(N, tf.float32)), tf.int32)

        h = self.pre_ln(x)
        x2d = tf.reshape(h, (B, H, H, D))
        y = self.conv(x2d)
        y = tf.reshape(y, (B, N, D))

        x = x + self.residual_gate * y

        head_outputs = [head(x) for head in self.cheap_heads]
        mixed = tf.concat(head_outputs, axis=-1)

        if self.projection:
            mixed = self.output_proj(mixed)

        return x + self.residual_gate * mixed

# --- VoidMixer Model Builder ---
def build_void_mixer_colorectal(input_shape=(75, 75, 3),
                         num_blocks=NUM_BLOCKS,
                         op_names=["std_norm", "scale_large", "binarize", "abs"],
                         embed_dim=EMBED_DIM,
                         use_aug=False):

    op_specs = auto_balance_ops(op_names, embed_dim)
    inputs = layers.Input(shape=input_shape)
    #x = layers.Lambda(lambda x: x + tf.random.normal(tf.shape(x), mean=0.0, stddev=0.01, output_shape=lambda s: s))(inputs)
    x = PatchEmbedding(PATCH_SIZE, embed_dim)(inputs)

    for _ in range(num_blocks):
        x = MultiHeadCheapOpBlock(embed_dim, op_specs)(x)

    x_max = layers.MaxPooling1D(pool_size=3, strides=1, padding="same")(x)
    x_avg = layers.AveragePooling1D(pool_size=3, strides=1, padding='same')(x)
    x = layers.Concatenate()([x_max, x_avg])
    x = layers.Dense(embed_dim // 4, activation="gelu")(x)

    #x = layers.GlobalAveragePooling1D()(x)   # (None, 2*dim)
    x = layers.Flatten()(x)
    x = layers.Dropout(0.00)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return models.Model(inputs, outputs)

# --- Build, Compile, Train ---
initial_lr = 1e-3
decay_steps = 10000
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=initial_lr,
    decay_steps=decay_steps,
    alpha=0.0
)
optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4)

model = build_void_mixer_colorectal()
model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()
model.fit(train_ds, epochs=100, validation_data=val_ds, batch_size=128)
model.evaluate(test_ds)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 75, 75, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embedding_3   │ (None, 225, 64)   │      4,864 │ input_layer_3[0]… │
│ (PatchEmbedding)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_cheap_o… │ (None, 225, 64)   │     18,241 │ patch_embedding_… │
│ (MultiHeadCheapOpB… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_cheap_o… │ (None, 225, 64)   │     18,241 │ multi_head_cheap… │
│ (MultiHeadCheapOpB… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_3     │ (None, 225, 64)   │          0 │ multi_head_cheap… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ average_pooling1d_3 │ (None, 225, 64)   │          0 │ multi_head_cheap… │
│ (AveragePooling1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 225, 128)  │          0 │ max_pooling1d_3[… │
│ (Concatenate)       │                   │            │ average_pooling1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_46 (Dense)    │ (None, 225, 16)   │      2,064 │ concatenate_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 3600)      │          0 │ dense_46[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 3600)      │          0 │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_47 (Dense)    │ (None, 8)         │     28,808 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 72,218 (282.10 KB)

 Trainable params: 72,218 (282.10 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 21s 135ms/step - accuracy: 0.2710 - loss: 3.2921 - val_accuracy: 0.4180 - val_loss: 1.5438
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.3843 - loss: 1.6489 - val_accuracy: 0.4720 - val_loss: 1.4039
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.4900 - loss: 1.4314 - val_accuracy: 0.5340 - val_loss: 1.2681
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5404 - loss: 1.2621 - val_accuracy: 0.5800 - val_loss: 1.1138
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6233 - loss: 0.9063 - val_accuracy: 0.6780 - val_loss: 0.8295
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6746 - loss: 0.8232 - val_accuracy: 0.4580 - val_loss: 1.2943
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6269 - loss: 0.9284 - val_accuracy: 0.6980 - val_loss: 0.7348
Epoch 8/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.7211 - loss: 0.6857 - val_accuracy: 

KeyboardInterrupt: 

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import tensorflow_datasets as tfds

# --- Hyperparameters ---
PATCH_SIZE = 5
EMBED_DIM = 48
NUM_BLOCKS = 2
IMG_SIZE = 75
NUM_CLASSES = 8

# --- Load & preprocess Colorectal Histology ---
def preprocess(example):
    image = tf.image.resize(example['image'], [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    label = example['label']
    return image, label

ds = tfds.load("colorectal_histology", split="train", as_supervised=False)
ds = ds.shuffle(5000, reshuffle_each_iteration=True).map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

# Split: 80% train, 10% val, 10% test
train_ds = ds.take(4000).batch(64).prefetch(tf.data.AUTOTUNE)
val_ds   = ds.skip(4000).take(500).batch(64).prefetch(tf.data.AUTOTUNE)
test_ds  = ds.skip(4500).take(500).batch(64).prefetch(tf.data.AUTOTUNE)

# --- Patch Embedding ---
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size, embed_dim):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size, padding='same')
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        return self.flatten(self.proj(x))

class ConvMixerBlock(layers.Layer):
    def __init__(self, embed_dim, kernel_size=5):
        super().__init__()
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.dwconv = layers.DepthwiseConv2D(kernel_size=kernel_size, padding='same', activation='gelu')
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.pwconv = layers.Conv2D(embed_dim, kernel_size=1, padding='same')
        self.activation = layers.Activation('gelu')

    def call(self, x):
        B = tf.shape(x)[0]
        N = x.shape[1]
        D = x.shape[2]
        H = tf.cast(tf.sqrt(tf.cast(N, tf.float32)), tf.int32)

        # LayerNorm -> Depthwise Conv -> Residual
        h = self.ln1(x)
        h2d = tf.reshape(h, (B, H, H, D))
        h2d = self.dwconv(h2d)
        h2d = tf.reshape(h2d, (B, N, D))
        x = x + h2d  # Residual after depthwise

        # LayerNorm -> Pointwise Conv (channel mixing)
        h = self.ln2(x)
        h2d = tf.reshape(h, (B, H, H, D))
        h2d = self.pwconv(h2d)
        h2d = self.activation(h2d)
        h2d = tf.reshape(h2d, (B, N, D))
        x = h2d  # No residual after pointwise (as in paper)

        return x


# --- Build ConvMixer Model ---
def build_convmixer(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_blocks=NUM_BLOCKS, embed_dim=EMBED_DIM):
    inputs = layers.Input(shape=input_shape)
    x = PatchEmbedding(PATCH_SIZE, embed_dim)(inputs)

    for _ in range(num_blocks):
        x = ConvMixerBlock(embed_dim)(x)

    # Global pooling + classification
    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    return models.Model(inputs, outputs)

# --- Compile & Train ---
initial_lr = 1e-3
decay_steps = 10000
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=initial_lr,
    decay_steps=decay_steps,
    alpha=0.0
)
optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4)

model = build_convmixer()
model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()
model.fit(train_ds, epochs=100, validation_data=val_ds)
model.evaluate(test_ds)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 75, 75, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ patch_embedding_3               │ (None, 225, 48)        │         3,648 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_mixer_block                │ (None, 225, 48)        │         3,792 │
│ (ConvMixerBlock)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_mixer_block_1              │ (None, 225, 48)        │         3,792 │
│ (ConvMixerBlock)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 48)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ (None, 8)              │           392 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,624 (45.41 KB)

 Trainable params: 11,624 (45.41 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 17s 97ms/step - accuracy: 0.2988 - loss: 1.7589 - val_accuracy: 0.5220 - val_loss: 1.1289
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.5617 - loss: 1.0381 - val_accuracy: 0.6160 - val_loss: 0.9161
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.6012 - loss: 0.9299 - val_accuracy: 0.5760 - val_loss: 0.9654
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.6187 - loss: 0.8927 - val_accuracy: 0.6380 - val_loss: 0.9395
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6453 - loss: 0.8905 - val_accuracy: 0.6480 - val_loss: 0.8063
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6669 - loss: 0.8374 - val_accuracy: 0.7420 - val_loss: 0.7328
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7029 - loss: 0.7670 - val_accuracy: 0.6940 - val_loss: 0.7203
Epoch 8/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7181 - loss: 0.7148 - val_accuracy: 0

[0.16060832142829895, 0.9440000057220459]

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import tensorflow_datasets as tfds

# --- Hyperparameters ---
PATCH_SIZE = 5
EMBED_DIM = 96  # Number of channels
NUM_BLOCKS = 2
IMG_SIZE = 75
NUM_CLASSES = 8

# --- Load & preprocess Colorectal Histology ---
def preprocess(example):
    image = tf.image.resize(example['image'], [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    label = example['label']
    return image, label

ds = tfds.load("colorectal_histology", split="train", as_supervised=False)
ds = ds.shuffle(5000, reshuffle_each_iteration=True).map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

# Split: 80% train, 10% val, 10% test
train_ds = ds.take(4000).batch(64).prefetch(tf.data.AUTOTUNE)
val_ds   = ds.skip(4000).take(500).batch(64).prefetch(tf.data.AUTOTUNE)
test_ds  = ds.skip(4500).take(500).batch(64).prefetch(tf.data.AUTOTUNE)

# --- Patch Embedding ---
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size, embed_dim):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size, padding='same')
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        return self.flatten(self.proj(x))

# --- Depthwise-only ConvMixer Block ---
class DepthwiseOnlyBlock(layers.Layer):
    def __init__(self, embed_dim, kernel_size=5):
        super().__init__()
        self.ln = layers.LayerNormalization(epsilon=1e-6)
        self.dwconv = layers.DepthwiseConv2D(kernel_size=kernel_size, padding='same', activation='gelu')

    def call(self, x):
        B = tf.shape(x)[0]
        N = x.shape[1]
        D = x.shape[2]
        H = tf.cast(tf.sqrt(tf.cast(N, tf.float32)), tf.int32)

        h = self.ln(x)
        h2d = tf.reshape(h, (B, H, H, D))
        h2d = self.dwconv(h2d)
        h2d = tf.reshape(h2d, (B, N, D))

        x = x + h2d  # Residual after depthwise
        return x

# --- Build Depthwise-only Model ---
def build_depthwise_only_convmixer(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_blocks=NUM_BLOCKS, embed_dim=EMBED_DIM):
    inputs = layers.Input(shape=input_shape)
    x = PatchEmbedding(PATCH_SIZE, embed_dim)(inputs)

    for _ in range(num_blocks):
        x = DepthwiseOnlyBlock(embed_dim)(x)

    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    return models.Model(inputs, outputs)

# --- Compile & Train ---
initial_lr = 1e-3
decay_steps = 10000
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=initial_lr,
    decay_steps=decay_steps,
    alpha=0.0
)
optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4)

model = build_depthwise_only_convmixer()
model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()
model.fit(train_ds, epochs=100, validation_data=val_ds)
model.evaluate(test_ds)


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 75, 75, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ patch_embedding_8               │ (None, 225, 96)        │         7,296 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_only_block_2          │ (None, 225, 96)        │         2,688 │
│ (DepthwiseOnlyBlock)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_only_block_3          │ (None, 225, 96)        │         2,688 │
│ (DepthwiseOnlyBlock)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 96)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_74 (Dense)                │ (None, 8)              │           776 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,448 (52.53 KB)

 Trainable params: 13,448 (52.53 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 76ms/step - accuracy: 0.3029 - loss: 1.7296 - val_accuracy: 0.4800 - val_loss: 1.1245
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.5238 - loss: 1.0934 - val_accuracy: 0.5080 - val_loss: 1.0202
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.5532 - loss: 0.9798 - val_accuracy: 0.6180 - val_loss: 0.9257
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5998 - loss: 0.9654 - val_accuracy: 0.6340 - val_loss: 0.9392
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.5930 - loss: 0.9159 - val_accuracy: 0.6240 - val_loss: 0.9743
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6180 - loss: 0.8988 - val_accuracy: 0.6120 - val_loss: 0.8873
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6570 - loss: 0.8425 - val_accuracy: 0.6220 - val_loss: 0.8730
Epoch 8/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.6645 - loss: 0.8405 - val_accuracy: 0

[0.33568865060806274, 0.8579999804496765]

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# --- Hyperparameters ---
PATCH_SIZE = 5
NUM_PATCHES = (75 // PATCH_SIZE) ** 2
EMBED_DIM = 256
NUM_BLOCKS = 2

import tensorflow_datasets as tfds

# --- Constants for new dataset ---
IMG_SIZE = 75
NUM_CLASSES = 8

# --- Load & preprocess CIFAR-10 ---
def preprocess_cifar10(example):
    image = tf.cast(example['image'], tf.float32) / 255.0  # normalize
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])   # resize to 75x75
    label = example['label']
    return image, label

ds = tfds.load("cifar10", split="train", as_supervised=False)
ds = ds.shuffle(50000, reshuffle_each_iteration=True).map(preprocess_cifar10, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = ds.take(45000).batch(64).prefetch(tf.data.AUTOTUNE)
val_ds   = ds.skip(45000).take(5000).batch(64).prefetch(tf.data.AUTOTUNE)

test_ds  = tfds.load("cifar10", split="test", as_supervised=False)
test_ds  = test_ds.map(preprocess_cifar10).batch(64).prefetch(tf.data.AUTOTUNE)


# --- Cheap Operations Dictionary ---
def cheap_ops_dict(embed_dim):
    return {
        "identity": layers.Lambda(lambda x: x),
        "softmax_gate": layers.Lambda(lambda x: x * tf.nn.softmax(x, axis=1)),
        "zeros": layers.Lambda(lambda x: tf.zeros_like(x)),
        "noise_injection": layers.Lambda(lambda x: x + tf.random.normal(tf.shape(x), stddev=0.01)),
        "spatial_dropout": layers.SpatialDropout1D(0.1),
        "scale_small": layers.Lambda(lambda x: 0.1 * x),
        "scale_large": layers.Lambda(lambda x: 10.0 * x),
        "negate": layers.Lambda(lambda x: -x),
        "abs": layers.Lambda(lambda x: tf.abs(x)),
        "square": layers.Lambda(lambda x: tf.square(x)),
        "sqrt": layers.Lambda(lambda x: tf.sqrt(tf.abs(x) + 1e-5)),
        "sign": layers.Lambda(lambda x: tf.sign(x)),
        "binarize": layers.Lambda(lambda x: tf.cast(x > 0.0, tf.float32)),
        "tanh_gate": layers.Lambda(lambda x: x * tf.tanh(x)),
        "sigmoid_gate": layers.Lambda(lambda x: x * tf.nn.sigmoid(x)),
        "mean_subtract": layers.Lambda(lambda x: x - tf.reduce_mean(x, axis=1, keepdims=True)),
        "std_norm": layers.Lambda(lambda x: (x - tf.reduce_mean(x, axis=1, keepdims=True)) /
                                               (tf.math.reduce_std(x, axis=1, keepdims=True) + 1e-5)),
        "global_context": layers.Lambda(lambda x: tf.repeat(tf.reduce_mean(x, axis=1, keepdims=True),
                                                            repeats=tf.shape(x)[1], axis=1)),
        "reverse": layers.Lambda(lambda x: tf.reverse(x, axis=[1])),
        "roll": layers.Lambda(lambda x: tf.roll(x, shift=1, axis=1)),
        #"channel_shuffle": layers.Lambda(lambda x: tf.random.shuffle(tf.transpose(x, perm=[0,2,1])) if tf.executing_eagerly() else x),
        "slice_first_half": layers.Lambda(lambda x: x[:, :, :x.shape[-1] // 2]),
        "slice_second_half": layers.Lambda(lambda x: x[:, :, x.shape[-1] // 2:]),
        "linear_skip": models.Sequential([
            layers.Dense(embed_dim),
            layers.Activation('linear')
        ]),
    }

# --- Helper: Auto-balance Head Dims ---
def auto_balance_ops(op_names, embed_dim):
    n = len(op_names)
    base = embed_dim // n
    rem = embed_dim % n
    return [(name, base + (1 if i < rem else 0)) for i, name in enumerate(op_names)]

# --- Patch Embedding ---
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size, embed_dim):
        super().__init__()
        self.projection = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size, padding='same')
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):
        return self.flatten(self.projection(x))

# --- Multi-Head CheapOp Block ---
class MultiHeadCheapOpBlockWeighted(layers.Layer):
    def __init__(self, embed_dim, op_specs, projection=True, top_k=4, l1_lambda=0.01):
        super().__init__()
        self.embed_dim = embed_dim
        self.projection = projection
        self.top_k = top_k  # number of active ops
        self.pre_ln = layers.LayerNormalization(epsilon=1e-6)
        self.conv = layers.DepthwiseConv2D(kernel_size=5, padding='same', activation='gelu')
        self.op_specs = op_specs
        self.cheap_heads = [cheap_ops_dict(head_dim)[name] for name, head_dim in op_specs]

        # L1-regularized gates
        self.head_gates = [
            self.add_weight(
                shape=(),
                initializer="ones",
                trainable=True,
                name=f"gate_{name}",
                regularizer=regularizers.l1(l1_lambda)
            )
            for name, _ in op_specs
        ]

        self.output_proj = layers.Dense(embed_dim) if projection else None
        self.residual_gate = self.add_weight(shape=(), initializer="ones", trainable=True, name="residual_gate")
        self.active_mask = tf.Variable([1.0]*len(op_specs), trainable=False, dtype=tf.float32)

    def update_active_mask(self):
        """Select top-k gates dynamically."""
        gates = tf.stack(self.head_gates)
        top_idx = tf.argsort(gates, direction='DESCENDING')[:self.top_k]
        mask = tf.zeros_like(gates)
        mask = tf.tensor_scatter_nd_update(mask, tf.expand_dims(top_idx, 1), tf.ones_like(top_idx, dtype=tf.float32))
        self.active_mask.assign(mask)

    def call(self, x):
        # Update mask on each call
        self.update_active_mask()

        B = tf.shape(x)[0]
        N = tf.shape(x)[1]
        D = x.shape[-1]
        H = tf.cast(tf.sqrt(tf.cast(N, tf.float32)), tf.int32)

        h = self.pre_ln(x)
        x2d = tf.reshape(h, (B,H,H,D))
        y = self.conv(x2d)
        y = tf.reshape(y, (B,N,D))
        x = x + self.residual_gate * y

        head_outputs = []
        for i, (head, gate) in enumerate(zip(self.cheap_heads, self.head_gates)):
            head_outputs.append(head(x) * gate * self.active_mask[i])
        mixed = tf.concat(head_outputs, axis=-1)
        if self.projection:
            mixed = self.output_proj(mixed)
        return x + self.residual_gate * mixed
class MultiHeadCheapOpBlockWeighted(layers.Layer):
    def __init__(self, embed_dim, op_specs, projection=True, top_k=4, l1_lambda=0.01):
        super().__init__()
        self.embed_dim = embed_dim
        self.projection = projection
        self.top_k = top_k  # number of active ops
        self.pre_ln = layers.LayerNormalization(epsilon=1e-6)
        self.conv = layers.DepthwiseConv2D(kernel_size=5, padding='same', activation='gelu')
        self.op_specs = op_specs
        self.cheap_heads = [cheap_ops_dict(head_dim)[name] for name, head_dim in op_specs]

        # L1-regularized gates
        self.head_gates = [
            self.add_weight(
                shape=(),
                initializer="ones",
                trainable=True,
                name=f"gate_{name}",
                regularizer=regularizers.l1(l1_lambda)
            )
            for name, _ in op_specs
        ]

        self.output_proj = layers.Dense(embed_dim) if projection else None
        self.residual_gate = self.add_weight(shape=(), initializer="ones", trainable=True, name="residual_gate")
        self.active_mask = tf.Variable([1.0]*len(op_specs), trainable=False, dtype=tf.float32)

    def update_active_mask(self):
        """Select top-k gates dynamically."""
        gates = tf.stack(self.head_gates)
        top_idx = tf.argsort(gates, direction='DESCENDING')[:self.top_k]
        mask = tf.zeros_like(gates)
        mask = tf.tensor_scatter_nd_update(mask, tf.expand_dims(top_idx, 1), tf.ones_like(top_idx, dtype=tf.float32))
        self.active_mask.assign(mask)

    def call(self, x):
        # Update mask on each call
        self.update_active_mask()

        B = tf.shape(x)[0]
        N = tf.shape(x)[1]
        D = x.shape[-1]
        H = tf.cast(tf.sqrt(tf.cast(N, tf.float32)), tf.int32)

        h = self.pre_ln(x)
        x2d = tf.reshape(h, (B,H,H,D))
        y = self.conv(x2d)
        y = tf.reshape(y, (B,N,D))
        x = x + self.residual_gate * y

        head_outputs = []
        for i, (head, gate) in enumerate(zip(self.cheap_heads, self.head_gates)):
            head_outputs.append(head(x) * gate * self.active_mask[i])
        mixed = tf.concat(head_outputs, axis=-1)
        if self.projection:
            mixed = self.output_proj(mixed)
        return x + self.residual_gate * mixed


# --- VoidMixer Model Builder ---
def build_void_mixer_colorectal(input_shape=(75, 75, 3),
                         num_blocks=NUM_BLOCKS,
                         op_names=["std_norm", "scale_large", "sign", "abs"],
                         embed_dim=EMBED_DIM,
                         use_aug=False):

    op_specs = auto_balance_ops(op_names, embed_dim)
    inputs = layers.Input(shape=input_shape)
    #x = layers.Lambda(lambda x: x + tf.random.normal(tf.shape(x), mean=0.0, stddev=0.01, output_shape=lambda s: s))(inputs)
    x = PatchEmbedding(PATCH_SIZE, embed_dim)(inputs)

    for _ in range(num_blocks):
        x = MultiHeadCheapOpBlockWeighted(embed_dim, op_specs)(x)

    x_max = layers.MaxPooling1D(pool_size=3, strides=1, padding="same")(x)
    x_avg = layers.AveragePooling1D(pool_size=3, strides=1, padding='same')(x)
    x = layers.Concatenate()([x_max, x_avg])
    x = layers.Dense(embed_dim // 4, activation="gelu")(x)

    #x = layers.GlobalAveragePooling1D()(x)   # (None, 2*dim)
    x = layers.Flatten()(x)
    x = layers.Dropout(0.00)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return models.Model(inputs, outputs)

# --- Build, Compile, Train ---
initial_lr = 1e-3
decay_steps = 10000
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=initial_lr,
    decay_steps=decay_steps,
    alpha=0.0
)
optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4)

model = build_void_mixer_colorectal()
model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()
model.fit(train_ds, epochs=100, validation_data=val_ds, batch_size=128)
model.evaluate(test_ds)


NameError: name 'regularizers' is not defined